In [39]:
import pandas as pd
import numpy as np

file_path = 'multi_platform_social_sentiment_evolution.csv'

try:
    df = pd.read_csv(file_path)
    print("--- Đọc file thành công! ---")
except FileNotFoundError:
    print("Không tìm thấy file. Hãy kiểm tra lại đường dẫn.")

print("\nThông tin Dataset:")
print(df.info())


--- Đọc file thành công! ---

Thông tin Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 31 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   post_id                           150000 non-null  object 
 1   platform                          150000 non-null  object 
 2   timestamp                         150000 non-null  object 
 3   date                              150000 non-null  object 
 4   hour_of_day                       150000 non-null  int64  
 5   day_of_week                       150000 non-null  int64  
 6   is_weekend                        150000 non-null  int64  
 7   user_id                           150000 non-null  object 
 8   followers                         150000 non-null  int64  
 9   account_age_days                  150000 non-null  int64  
 10  verified                          150000 non-null  int64  
 11  top

In [40]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Extract additional time features (optional)
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
df['month'] = df['timestamp'].dt.month

In [41]:
df['log_followers'] = np.log1p(df['followers'])

In [42]:
# 6. DEFINE FEATURE GROUP
categorical_cols = [
    'platform',
    'topic',
    'language',
    'media_type',
    'sentiment_category',
    'location'
]

numerical_cols = [
    'log_followers',
    'account_age_days',
    'content_length',
    'num_hashtags',
    'sentiment_positive',
    'sentiment_negative',
    'sentiment_neutral',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'hours_since_post',
    'toxicity_score'
]


In [43]:
# Target chính
df['target_engagement'] = df['engagement_rate_per_1k_followers']

In [44]:
# dùng percentile để chia high / low engagement
threshold = df['target_engagement'].quantile(0.8)

df['label'] = (df['target_engagement'] > threshold).astype(int)

print("Threshold:", threshold)
print(df['label'].value_counts())

Threshold: 40.0
label
0    120012
1     29988
Name: count, dtype: int64


In [45]:
drop_cols = [
    'post_id',
    'timestamp',
]

df = df.drop(columns=drop_cols)

In [46]:
leakage_cols = [
    'likes',
    'shares',
    'comments',
    'total_engagement',
    'views',
    'engagement_rate_per_1k_followers',
    'viral_coefficient'
]

df = df.drop(columns=leakage_cols)

print("Remaining columns:", df.columns)

Remaining columns: Index(['platform', 'date', 'hour_of_day', 'day_of_week', 'is_weekend',
       'user_id', 'followers', 'account_age_days', 'verified', 'topic',
       'language', 'content_length', 'media_type', 'num_hashtags',
       'sentiment_category', 'sentiment_positive', 'sentiment_negative',
       'sentiment_neutral', 'hours_since_post', 'cross_platform_spread',
       'toxicity_score', 'location', 'hour', 'day', 'month', 'log_followers',
       'target_engagement', 'label'],
      dtype='object')


In [47]:
y = df['label']

# X là các đặc trưng (Features). 
# Anh dùng errors='ignore' để nếu cột đã bị xóa trước đó thì code vẫn không báo lỗi.
X = df.drop(columns=['label', 'target_engagement'], errors='ignore')

In [48]:
# 3. Chia tập Train/Test bằng GroupShuffleSplit theo user_id
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['user_id']))

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

print(f"\n--- Đã chia dữ liệu xong ---")
print(f"Tập Train: {len(df_train)} dòng")
print(f"Tập Test: {len(df_test)} dòng")


--- Đã chia dữ liệu xong ---
Tập Train: 120070 dòng
Tập Test: 29930 dòng


In [49]:
import networkx as nx
from itertools import combinations

# 1. Khởi tạo Graph
G_user = nx.Graph()

# 2. Thêm nodes từ df_train (đã sửa tên biến từ X_train_df thành df_train)
users_train = df_train['user_id'].unique()
G_user.add_nodes_from(users_train)

print(f"Total users in Train: {len(users_train)}")

# 3. Hàm xây dựng cạnh (Đảm bảo dùng đúng các cột trong df_train)
def build_user_graph_from_context(data_source):
    print("Đang tạo các kết nối User-User...")
    # Gom nhóm theo các tiêu chí bạn muốn (location, topic, thời gian)
    groups = data_source.groupby(['date', 'hour_of_day', 'topic', 'location'])
    
    for name, group in groups:
        user_list = group['user_id'].unique()
        if 1 < len(user_list) < 50: 
            for u1, u2 in combinations(user_list, 2):
                if G_user.has_edge(u1, u2):
                    G_user[u1][u2]['weight'] += 1
                else:
                    G_user.add_edge(u1, u2, weight=1)

# 4. Gọi hàm truyền vào df_train
build_user_graph_from_context(df_train)

print(f"Total edges created: {G_user.number_of_edges()}")

Total users in Train: 103644
Đang tạo các kết nối User-User...
Total edges created: 70046


In [50]:
import numpy as np



# Bây giờ mới chạy lệnh xuất file Nodes
nodes_df = pd.DataFrame({'Id': list(G_user.nodes())})

node_info = df_train.groupby('user_id').agg({
    'platform': 'first',
    'location': 'first',
    'log_followers': 'mean',
    'topic': 'first'
}).reset_index()

node_info.rename(columns={'user_id': 'Id'}, inplace=True)
nodes_df = nodes_df.merge(node_info, on='Id', how='left')

nodes_df.to_csv("nodes_cross_platform.csv", index=False)
print("Đã xuất file nodes thành công!")

Đã xuất file nodes thành công!


In [51]:



# ==========================================
# 2. XUẤT FILE EDGES (Edges.csv)
# ==========================================
edges_list = []

# Duyệt qua các cạnh trong đồ thị để lấy Source, Target và Weight
for u, v, data in G_user.edges(data=True):
    edges_list.append({
        'Source': u,
        'Target': v,
        'Weight': data.get('weight', 1),
        'Type': 'Undirected' # Đồ thị vô hướng cho tính chất "xuất hiện chung"
    })

edges_df = pd.DataFrame(edges_list)

# Lưu file Edges
edges_df.to_csv("edges_cross_platform.csv", index=False)
print(f"Đã xuất {len(edges_df)} edges vào file edges_cross_platform.csv")

Đã xuất 70046 edges vào file edges_cross_platform.csv


In [53]:
print(df_train.columns)
print (df_test.columns)

Index(['platform', 'date', 'hour_of_day', 'day_of_week', 'is_weekend',
       'user_id', 'followers', 'account_age_days', 'verified', 'topic',
       'language', 'content_length', 'media_type', 'num_hashtags',
       'sentiment_category', 'sentiment_positive', 'sentiment_negative',
       'sentiment_neutral', 'hours_since_post', 'cross_platform_spread',
       'toxicity_score', 'location', 'hour', 'day', 'month', 'log_followers',
       'target_engagement', 'label'],
      dtype='object')
Index(['platform', 'date', 'hour_of_day', 'day_of_week', 'is_weekend',
       'user_id', 'followers', 'account_age_days', 'verified', 'topic',
       'language', 'content_length', 'media_type', 'num_hashtags',
       'sentiment_category', 'sentiment_positive', 'sentiment_negative',
       'sentiment_neutral', 'hours_since_post', 'cross_platform_spread',
       'toxicity_score', 'location', 'hour', 'day', 'month', 'log_followers',
       'target_engagement', 'label'],
      dtype='object')


In [54]:
# 1. Đọc file kết quả từ Gephi
df_gephi = pd.read_csv('gephi-result-3.csv')

# 2. Chọn các đặc trưng 'đắt giá' nhất cho mô hình
graph_features = [
    'pageranks', 'modularity_class', 'Degree', 
    'Weighted Degree', 'betweenesscentrality', 
    'eigencentrality', 'clustering'
]

# 3. Nối vào tập dữ liệu gốc (df_train và df_test)
# Id trong Gephi tương ứng với user_id trong dữ liệu của bạn
df_gephi_subset = df_gephi[['Id'] + graph_features]

# Merge vào df_train
df_train = df_train.merge(df_gephi_subset, left_on='user_id', right_on='Id', how='left').fillna(0)
# Merge vào df_test
df_test = df_test.merge(df_gephi_subset, left_on='user_id', right_on='Id', how='left').fillna(0)

# Xóa cột Id thừa sau khi merge
df_train.drop(columns=['Id'], inplace=True)
df_test.drop(columns=['Id'], inplace=True)

In [55]:
df_train.head()

,platform,date,hour_of_day,day_of_week,is_weekend,user_id,followers,account_age_days,verified,topic,...,log_followers,target_engagement,label,pageranks,modularity_class,Degree,Weighted Degree,betweenesscentrality,eigencentrality,clustering
0,TikTok,2025-04-19,1,5,1,user_426711,137,306,0,Finance,...,4.927254,0.00,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,Twitter,2025-04-19,5,5,1,user_221610,1974,2310,0,Food,...,7.588324,1.52,0,0.000002,29228.0,0.0,0.0,0.0,0.0,0.0
2,Instagram,2025-04-19,6,5,1,user_313440,1366,2057,0,Education,...,7.220374,71.74,1,0.000002,11523.0,0.0,0.0,0.0,0.0,0.0
3,Reddit,2025-04-19,7,5,1,user_19777,124,1400,0,Climate,...,4.828314,120.97,1,0.000002,43658.0,0.0,0.0,0.0,0.0,0.0
4,Facebook,2025-04-19,8,5,1,user_157662,265,57,0,Finance,...,5.583496,0.00,0,0.000002,31693.0,0.0,0.0,0.0,0.0,0.0
